In [9]:
from pathlib import Path
import numpy as np
import json
import time
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.ensemble import IsolationForest

In [10]:
ROOT = Path(r"E:/marcel.furs/Documents/Data_Run_Walk/Data_Run_Walk")
N_SPLITS = 30

In [11]:
from sklearn.svm import OneClassSVM

def evaluate_ocsvm(X_train_raw, X_val_raw, X_test_normal_raw, X_test_anomaly_raw,
                   seed, kernel="rbf", gamma="scale", nu=0.5):
    X_train         = X_train_raw.reshape(X_train_raw.shape[0], -1)
    X_test_normal   = X_test_normal_raw.reshape(X_test_normal_raw.shape[0], -1)
    X_test_anomaly  = X_test_anomaly_raw.reshape(X_test_anomaly_raw.shape[0], -1)
    scaler = StandardScaler()
    X_train_norm        = scaler.fit_transform(X_train)
    X_test_normal_norm  = scaler.transform(X_test_normal)
    X_test_anomaly_norm = scaler.transform(X_test_anomaly)

    # OneClassSVM nie ma random_state, ale podajemy seed dla spójności (no-op tu)
    ocsvm = OneClassSVM(kernel=kernel, gamma=gamma, nu=nu)
    ocsvm.fit(X_train_norm)
    # niższy decision_function = bardziej anomaly, więc neguj
    score_normal  = -ocsvm.decision_function(X_test_normal_norm)
    score_anomaly = -ocsvm.decision_function(X_test_anomaly_norm)

    y_true  = np.concatenate([np.zeros(len(score_normal)), np.ones(len(score_anomaly))])
    y_score = np.concatenate([score_normal, score_anomaly])
    return {
        "auc": float(roc_auc_score(y_true, y_score)),
        "ap":  float(average_precision_score(y_true, y_score)),
        "n_train": len(X_train_norm),
        "n_val": 0,
        "n_test_normal": len(X_test_normal_norm),
        "n_test_anomaly": len(X_test_anomaly_norm),
    }

In [12]:
from sklearn.decomposition import PCA

def evaluate_pca(X_train_raw, X_val_raw, X_test_normal_raw, X_test_anomaly_raw, seed,
                 n_components=2):
    X_train         = X_train_raw.reshape(X_train_raw.shape[0], -1)
    X_test_normal   = X_test_normal_raw.reshape(X_test_normal_raw.shape[0], -1)
    X_test_anomaly  = X_test_anomaly_raw.reshape(X_test_anomaly_raw.shape[0], -1)

    scaler = StandardScaler()
    X_train_norm        = scaler.fit_transform(X_train)
    X_test_normal_norm  = scaler.transform(X_test_normal)
    X_test_anomaly_norm = scaler.transform(X_test_anomaly)

    pca = PCA(n_components=n_components, random_state=seed)
    pca.fit(X_train_norm)

    # Reconstruction
    X_test_normal_recon  = pca.inverse_transform(pca.transform(X_test_normal_norm))
    X_test_anomaly_recon = pca.inverse_transform(pca.transform(X_test_anomaly_norm))

    # Reconstruction error (anomaly score)
    score_normal  = np.mean((X_test_normal_norm  - X_test_normal_recon)**2,  axis=1)
    score_anomaly = np.mean((X_test_anomaly_norm - X_test_anomaly_recon)**2, axis=1)

    y_true  = np.concatenate([np.zeros(len(score_normal)), np.ones(len(score_anomaly))])
    y_score = np.concatenate([score_normal, score_anomaly])

    return {
        "auc": float(roc_auc_score(y_true, y_score)),
        "ap":  float(average_precision_score(y_true, y_score)),
        "n_train": len(X_train_norm),
        "n_val": 0,
        "n_test_normal": len(X_test_normal_norm),
        "n_test_anomaly": len(X_test_anomaly_norm),
    }

In [13]:
from sklearn.ensemble import IsolationForest

def evaluate_iforest(X_train_raw, X_val_raw, X_test_normal_raw, X_test_anomaly_raw,
                     seed, n_estimators=100, contamination="auto"):
    X_train         = X_train_raw.reshape(X_train_raw.shape[0], -1)
    X_test_normal   = X_test_normal_raw.reshape(X_test_normal_raw.shape[0], -1)
    X_test_anomaly  = X_test_anomaly_raw.reshape(X_test_anomaly_raw.shape[0], -1)
    scaler = StandardScaler()
    X_train_norm        = scaler.fit_transform(X_train)
    X_test_normal_norm  = scaler.transform(X_test_normal)
    X_test_anomaly_norm = scaler.transform(X_test_anomaly)
    
    iforest = IsolationForest(n_estimators=n_estimators, contamination=contamination,
                              random_state=seed)
    iforest.fit(X_train_norm)
    # niższy score_samples = bardziej anomaly, więc neguj
    score_normal  = -iforest.score_samples(X_test_normal_norm)
    score_anomaly = -iforest.score_samples(X_test_anomaly_norm)
    
    y_true  = np.concatenate([np.zeros(len(score_normal)), np.ones(len(score_anomaly))])
    y_score = np.concatenate([score_normal, score_anomaly])
    return {
        "auc": float(roc_auc_score(y_true, y_score)),
        "ap":  float(average_precision_score(y_true, y_score)),
        "n_train": len(X_train_norm),
        "n_val": 0,
        "n_test_normal": len(X_test_normal_norm),
        "n_test_anomaly": len(X_test_anomaly_norm),
    }

In [14]:
def make_split(X_normal_S1, subj_normal_S1, seed):
    unique_subj = np.unique(subj_normal_S1)
    n_val = max(1, int(len(unique_subj) * 0.2))
    rng = np.random.RandomState(seed)
    val_subj = rng.choice(unique_subj, size=n_val, replace=False)
    train_subj = np.array([s for s in unique_subj if s not in val_subj])
    mask_tr = np.isin(subj_normal_S1, train_subj)
    mask_va = np.isin(subj_normal_S1, val_subj)
    return X_normal_S1[mask_tr], X_normal_S1[mask_va]





import json
from datetime import datetime
from pathlib import Path

def save_result_v2(
    experiment,           # 'sex' / 'speed' / 'walk_run'
    direction,            # 'M_to_F' / etc.
    config,               # 'A' / 'B' / None (dla baseline'ów które nie mają config)
    model,                # 'dense_VAE' / 'PCA' / 'OC_SVM' / 'IF' / 'LSTM_VAE'
    hyperparameters,      # dict: np. {"n_components": 4} albo {"latent_dim": 4, "width": 128, ...}
    data_info,            # dict: {"root_exp": str, "s1_pool": int, "s2_normal": int, "s2_anomaly": int, "n_features": int}
    aucs, aps,            # listy 30 wartości
    seeds,                # lista użytych seedów
    duration_s,           # czas całego batcha
    notes="",             # opcjonalna notatka tekstowa
    results_path="results_v2.json"
):
    entry = {
        "experiment": experiment,
        "direction": direction,
        "config": config,
        "model": model,
        "hyperparameters": hyperparameters,
        "data_info": data_info,
        "n_splits": len(aucs),
        "seeds": list(seeds),
        "aucs": list(map(float, aucs)),
        "aps":  list(map(float, aps)),
        "auc_mean": float(np.mean(aucs)),
        "auc_std":  float(np.std(aucs)),
        "ap_mean":  float(np.mean(aps)),
        "ap_std":   float(np.std(aps)),
        "duration_s": float(duration_s),
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "notes": notes,
    }
    if Path(results_path).exists():
        with open(results_path) as f:
            all_results = json.load(f)
    else:
        all_results = []
    # dedup key uwzględnia też hyperparameters (różne n_components = różne wpisy)
    def hp_key(hp):
        return tuple(sorted(hp.items()))
    key = (experiment, direction, config, model, hp_key(hyperparameters))
    all_results = [r for r in all_results
                   if (r["experiment"], r["direction"], r["config"], r["model"],
                       hp_key(r["hyperparameters"])) != key]
    all_results.append(entry)
    with open(results_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f" zapisano: {experiment}/{direction}/{config or '-'}/{model}/{hp_key(hyperparameters)}")

In [15]:
MODEL_NAME = "OC_SVM"
HYPERPARAMS = {"kernel": "rbf", "gamma": "scale", "nu": 0.5}
CONFIG = None
NOTES = "One-Class SVM baseline with RBF kernel, sklearn defaults"

In [16]:
EXPERIMENTS = [
    ("sex",       "M_to_F",       "exp1a_train_M_detect_F"),
    ("sex",       "F_to_M",       "exp1b_train_F_detect_M"),
    ("speed",     "Slow_to_Fast", "exp_train_slow_detect_fast"),
    ("speed",     "Fast_to_Slow", "exp_train_fast_detect_slow"),
    ("walk_run",  "Walk_to_Run",  "exp_train_walk_detect_run"),
    ("walk_run",  "Run_to_Walk",  "exp_train_run_detect_walk"),
]
N_SPLITS = 30
SEEDS = list(range(N_SPLITS))


t_total = time.time()

for experiment, direction, root_exp in EXPERIMENTS:
    print(f"\n{'='*70}")
    print(f"  {experiment} / {direction}  (folder: {root_exp})")
    print('='*70)
    EXP_DIR = ROOT / "EXPERIMENTS" / root_exp

    # load
    X_train_paper  = np.load(EXP_DIR / "X_train.npy")
    X_val_paper    = np.load(EXP_DIR / "X_val.npy")
    X_test_normal  = np.load(EXP_DIR / "X_test_normal.npy")
    X_test_anomaly = np.load(EXP_DIR / "X_test_anomaly.npy")
    subj_train = np.load(EXP_DIR / "subjects_train.npy", allow_pickle=True)
    subj_val   = np.load(EXP_DIR / "subjects_val.npy",   allow_pickle=True)
    X_S1_normal    = np.concatenate([X_train_paper, X_val_paper])
    subj_S1_normal = np.concatenate([subj_train, subj_val])

    # data_info dict
    data_info = {
        "root_exp": root_exp,
        "s1_pool": int(X_S1_normal.shape[0]),
        "s2_normal": int(X_test_normal.shape[0]),
        "s2_anomaly": int(X_test_anomaly.shape[0]),
        "input_shape": list(X_S1_normal.shape[1:]),     # np. [100, 78]
        "n_features_flat": int(np.prod(X_S1_normal.shape[1:])),  # 7800
        "n_unique_subjects_s1": int(len(np.unique(subj_S1_normal))),
    }

    print(f"  S1 pool: {data_info['s1_pool']} cykli, "
          f"S2 normal: {data_info['s2_normal']}, anomaly: {data_info['s2_anomaly']}")

    # pętla N_SPLITS splitów
    results = []
    t0 = time.time()
    for s in SEEDS:
        X_tr, X_va = make_split(X_S1_normal, subj_S1_normal, seed=s)
        #r = evaluate_pca(X_tr, X_va, X_test_normal, X_test_anomaly,
        #                 seed=45, n_components=HYPERPARAMS["n_components"])
        #r = evaluate_iforest(X_tr, X_va, X_test_normal, X_test_anomaly,
        #             seed=s, n_estimators=HYPERPARAMS["n_estimators"])
        r = evaluate_ocsvm(X_tr, X_va, X_test_normal, X_test_anomaly,
                   seed=s,
                   kernel=HYPERPARAMS["kernel"],
                   gamma=HYPERPARAMS["gamma"],
                   nu=HYPERPARAMS["nu"])
        results.append(r)
    duration = time.time() - t0

    aucs = [r["auc"] for r in results]
    aps  = [r["ap"]  for r in results]
    print(f"  czas: {duration:.1f}s | "
          f"AUC: {np.mean(aucs):.3f} ± {np.std(aucs):.3f} | "
          f"AP: {np.mean(aps):.3f} ± {np.std(aps):.3f}")
    
    save_result_v2(
        experiment=experiment,
        direction=direction,
        config=CONFIG,
        model=MODEL_NAME,
        hyperparameters=HYPERPARAMS,
        data_info=data_info,
        aucs=aucs, aps=aps,
        seeds=SEEDS,
        duration_s=duration,
        notes=NOTES,
    )


print(f"\n{'='*70}\nTOTAL TIME: {time.time()-t_total:.1f}s")


  sex / M_to_F  (folder: exp1a_train_M_detect_F)
  S1 pool: 96 cykli, S2 normal: 94, anomaly: 235
  czas: 4.9s | AUC: 0.901 ± 0.007 | AP: 0.939 ± 0.005
 zapisano: sex/M_to_F/-/OC_SVM/(('gamma', 'scale'), ('kernel', 'rbf'), ('nu', 0.5))

  sex / F_to_M  (folder: exp1b_train_F_detect_M)
  S1 pool: 120 cykli, S2 normal: 115, anomaly: 198
  czas: 5.7s | AUC: 0.652 ± 0.023 | AP: 0.701 ± 0.018
 zapisano: sex/F_to_M/-/OC_SVM/(('gamma', 'scale'), ('kernel', 'rbf'), ('nu', 0.5))

  speed / Slow_to_Fast  (folder: exp_train_slow_detect_fast)
  S1 pool: 219 cykli, S2 normal: 197, anomaly: 417
  czas: 18.8s | AUC: 0.857 ± 0.010 | AP: 0.903 ± 0.008
 zapisano: speed/Slow_to_Fast/-/OC_SVM/(('gamma', 'scale'), ('kernel', 'rbf'), ('nu', 0.5))

  speed / Fast_to_Slow  (folder: exp_train_fast_detect_slow)
  S1 pool: 204 cykli, S2 normal: 213, anomaly: 416
  czas: 18.0s | AUC: 0.691 ± 0.023 | AP: 0.759 ± 0.022
 zapisano: speed/Fast_to_Slow/-/OC_SVM/(('gamma', 'scale'), ('kernel', 'rbf'), ('nu', 0.5))

  w

 zapisano: walk_run/Run_to_Walk/AA/PCA/(('n_components', 2),)
